# Atlas v1 — DP622 dynamic geometry-gated prioritization

This notebook runs computational reconstruction, published-control validation, and conditional candidate prioritization. Novel outputs are computational predictions requiring experimental validation. Use a **GPU runtime** because the pinned ThermoMPNN-D epistatic implementation requires CUDA.

In [ ]:
# Clone Atlas. Change ATLAS_REVISION only when intentionally reproducing another revision.
import os, subprocess
from pathlib import Path

ATLAS_URL = 'https://github.com/Noelduval/atlas-therapeutic-optimization.git'
ATLAS_REVISION = 'main'
ATLAS_DIR = Path('/content/Atlas')
if not ATLAS_DIR.exists():
    subprocess.run(['git', 'clone', ATLAS_URL, str(ATLAS_DIR)], check=True)
subprocess.run(['git', '-C', str(ATLAS_DIR), 'checkout', ATLAS_REVISION], check=True)
os.chdir(ATLAS_DIR)
print(subprocess.run(['git', 'rev-parse', 'HEAD'], text=True, capture_output=True, check=True).stdout.strip())

In [ ]:
# Install Atlas, OpenMM, and the dependencies declared by the official model notebooks/environments.
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{ATLAS_DIR}[dynamics]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'omegaconf', 'wandb', 'pytorch-lightning', 'scipy', 'scikit-learn', 'joblib'], check=True)

In [ ]:
# Clone the official predictors at the commits recorded by Atlas.
THERMOMPNN_REV = '2b04fd370e399911b1fa5848112cc9013f084110'
THERMOMPNN_D_REV = 'df9a75aaddb674a7c4c193005031fc0536d325fb'
EXTERNAL = ATLAS_DIR / '.external'
EXTERNAL.mkdir(exist_ok=True)
repos = [
    ('https://github.com/Kuhlman-Lab/ThermoMPNN.git', EXTERNAL / 'ThermoMPNN', THERMOMPNN_REV),
    ('https://github.com/Kuhlman-Lab/ThermoMPNN-D.git', EXTERNAL / 'ThermoMPNN-D', THERMOMPNN_D_REV),
]
for url, path, revision in repos:
    if not path.exists():
        subprocess.run(['git', 'clone', url, str(path)], check=True)
    subprocess.run(['git', '-C', str(path), 'checkout', revision], check=True)
    actual = subprocess.run(['git', '-C', str(path), 'rev-parse', 'HEAD'], text=True, capture_output=True, check=True).stdout.strip()
    assert actual == revision, (path, actual, revision)
print('Pinned model repositories are ready.')

In [ ]:
# ThermoMPNN-D epistatic inference is CUDA-only at the pinned revision.
import torch
print('torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, reconnect, and rerun.')

## Optional persistent storage
Uncomment the next cell to mount Drive. The default run remains under `/content/Atlas/outputs`, which is zipped for download at the end.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Run the exact production CLI. A failed validation gate raises and produces no novel artifacts.
command = [
    sys.executable, '-m', 'atlas', 'run',
    '--input', str(ATLAS_DIR / 'data/23WN.cif'),
    '--output-root', str(ATLAS_DIR / 'outputs'),
    '--thermompnn-repo', str(EXTERNAL / 'ThermoMPNN'),
    '--thermompnn-d-repo', str(EXTERNAL / 'ThermoMPNN-D'),
    '--dynamics-mode', 'minimize',
]
subprocess.run(command, cwd=ATLAS_DIR, check=True)
RUN_DIR = max((ATLAS_DIR / 'outputs').glob('run-*'), key=lambda path: path.stat().st_mtime)
print('Run directory:', RUN_DIR)

In [ ]:
# Review the benchmark before the candidates.
import pandas as pd
from IPython.display import display, Markdown, Image
display(pd.read_csv(RUN_DIR / 'known_mutation_validation.csv'))
display(Markdown((RUN_DIR / 'validation_report.md').read_text()))
display(Markdown((RUN_DIR / 'pipeline_warnings.md').read_text()))
display(Image(filename=str(RUN_DIR / 'figures/validation_dashboard.png')))
display(Image(filename=str(RUN_DIR / 'figures/catalytic_geometry_boxplots.png')))

In [ ]:
# These files exist only if the hard gate passed.
ranked = RUN_DIR / 'novel_candidates_ranked.csv'
if ranked.exists():
    display(pd.read_csv(ranked).head(20))
    display(Image(filename=str(RUN_DIR / 'figures/candidate_ranking_summary.png')))
else:
    print('Validation did not permit novel design; this is a valid negative result.')

In [ ]:
# Zip only this immutable run and download it.
import shutil
archive = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print(archive)
from google.colab import files
files.download(archive)